# ***Data Cleaning + Regressions + Visualizations***

In [4]:
# Import packages
import numpy as np
import pandas as pd
import os 
from scipy.stats import chi2_contingency
import scipy.stats as stats
import statsmodels.formula.api as smf
import plotly.express as px
from marginaleffects import predictions, avg_predictions, avg_slopes

In [5]:
# Set directory for raw data, download raw data
brfss_raw_dir = '{}/data/BRFSS2023.csv'.format(os.getcwd())
brfss_raw_df = pd.read_csv(brfss_raw_dir)

In [6]:
# Only select columns of interest
brfss_df = brfss_raw_df[['addepev3', 'firearm5', 'menthlth', '_age80', 'sexvar', '_racegr3', 'primins1', 'educa', 'income3', 'loadulk2', 'children', '_state']]

# Create a new data table from the subsetted data
brfss_dir = '{}/data/BRFSS_subset.csv'.format(os.getcwd())
brfss_df.to_csv(path_or_buf = brfss_dir, index=False)

In [7]:
# List columns and examine first few rows
print(brfss_df.columns.tolist())
brfss_df.head()

['addepev3', 'firearm5', 'menthlth', '_age80', 'sexvar', '_racegr3', 'primins1', 'educa', 'income3', 'loadulk2', 'children', '_state']


,addepev3,firearm5,menthlth,_age80,sexvar,_racegr3,primins1,educa,income3,loadulk2,children,_state
0,2.0,NaN,88.0,80,2,1.0,3.0,5.0,99.0,NaN,88.0,1
1,1.0,NaN,88.0,80,2,1.0,3.0,5.0,99.0,NaN,88.0,1
2,2.0,NaN,2.0,80,2,2.0,3.0,4.0,2.0,NaN,88.0,1
3,1.0,NaN,88.0,78,2,1.0,3.0,5.0,99.0,NaN,88.0,1
4,1.0,NaN,88.0,76,2,1.0,3.0,5.0,7.0,NaN,88.0,1


# **Clean Variables - addepev3, firearm5, loadulk2, _ment14d, _racegr3, primins1, educa, income3, children** 

In [8]:
# ADDEPEV3 (Depression): Set No (2) to No (0)
                       # Set Don't know (7) and Refused (9) to missing
brfss_df['addepev3'] = brfss_df['addepev3'].replace({2:0,
                                                     7: np.nan, 9: np.nan})

# FIREARM5 (Firearms): Set No (2) to No (0)
                     # Don't Know (7) and Refused (9) to missing
brfss_df["firearm5"] = brfss_df["firearm5"].replace({2:0,
                                                     7: np.nan, 9: np.nan})

# LOADULK2 (Unlocked loaded firearms): Set No (2) to No (0) 
                                     # Don't know (7) and Refused (9) to missing 
brfss_df["loadulk2"] = brfss_df["loadulk2"].replace({2:0,
                                                     7: np.nan, 9: np.nan})

# _AGE80 (Age collapsed above 80): No Missingness

# SEXVAR (Sex): No Missingness

# MENTHLTH (Days of Poor Mental Health): Set Don't know/Not Sure/Refused (9) to missing 
brfss_df["menthlth"] = brfss_df["menthlth"].replace({77: np.nan,
                                                     88: 0,
                                                    99: np.nan})

# _RACEGR3 (Racial group): Set Don't know/Not Sure/Refused (9) to missing
brfss_df["_racegr3"] = brfss_df["_racegr3"].replace({9: np.nan})

# PRIMINS1 (Insurance group): Set employer-sponsored (1) and other private (2) to private (1).
                            # Set Medicare (3), Medigap (4), Medicaid (5), and CHIP (6) to public (2). 
                            # Set military (7), Indian Health Services (8), state-sponsored (9), and other government (10) to other (3)
                            # Set no coverage (88) to uninsured (4) 
                            # Set Don't know/Not Sure (77) and Refused (99) to missing
brfss_df["primins1"] = brfss_df["primins1"].replace({2:1,
                                                     3:2, 4:2, 5:2, 6:2,
                                                     7:3, 8:3, 9:3, 10:3,
                                                     88:4,
                                                     77:np.nan, 99:np.nan})

# EDUCA (Education level): Set never attended (1), elementary (2), some high school (3), and high school/GED (4) to high school or less (1)
                         # Set some college/trade school (5) to some college (2) 
                         # Set 4-year degreee or more (6) to Bachelor's or more (3) 
                         # Set Refused (9) to missing  
brfss_df["educa"] = brfss_df["educa"].replace({2:1, 3:1, 4:1,
                                               5:2,
                                               6:3,
                                               9:np.nan})

# INCOME3 (Income categories): Set less than 10k (1), less than 15k (2), and less than 20k (3) to less than 20k (1)
                             # Set less than 25k (4), less than 35k (5), and less than 50k (6) to 20k - 50k (2) 
                             # Set less than 75k (7) and less than 100k (8) to 50k - 100k (3) 
                             # Set less than 150k (9) and less than 200k (10) to 100k - 200k (4)
                             # Set 200k or more (11) to 200k or more (5) 
                             # Set Don't know/Not sure (77) and Refused (99) to missing  
brfss_df["income3"] = brfss_df["income3"].replace({2:1, 3:1,
                                                   4:2, 5:2, 6:2,
                                                   7:3, 8:3,
                                                   9:4, 10:4,
                                                   11:5,
                                                   77:np.nan, 99: np.nan})

# CHILDREN (Children in home indicator): Set any positive number of children to 1. 
                                       # Set No children (88) to None (0)
                                       # Set Refused (99) to missing  
brfss_df.loc[brfss_df["children"].between(1, 87), "children"] = 1
brfss_df["children"] = brfss_df["children"].replace({88:0,
                                                     99:np.nan})

# _STATE (FIPS code): Change numeric labels to state abbreviations
brfss_df["_state"] = brfss_df["_state"].replace({1:'AL', 2:'AK', 4:'AZ', 5:'AR', 6:'CA', 8:'CO', 9:'CT', 10:'DE', 
                                                 11:'DC', 12:'FL', 13:'GA', 15:'HI', 16:'ID', 17:'IL', 18:'IN', 19:'IA', 20:'KS', 
                                                 22:'LA', 23:'ME', 24:'MD', 25:'MA', 26:'MI', 27:'MN', 28:'MS', 29:'MO', 30:'MT', 
                                                 31:'NE', 32:'NV', 33:'NH', 34:'NJ', 35:'NM', 36:'NY', 37:'NC', 38:'ND', 39:'OH', 40:'OK', 
                                                 41:'OR', 44:'RI', 45:'SC', 46:'SD', 47:'TN', 48:'TX', 49:'UT', 50:'VT', 
                                                 51:'VA', 53:'WA', 54:'WV', 55:'WI', 56:'WY',
                                                 66:'GU', 72:'PR', 78:'VI'})

C:\Users\samue\AppData\Local\Temp\ipykernel_31876\3881905558.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
C:\Users\samue\AppData\Local\Temp\ipykernel_31876\3881905558.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
C:\Users\samue\AppData\Local\Temp\ipykernel_31876\3881905558.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-do

In [34]:
# Copy cleaned version of dataset
brfss_clean_df = brfss_df.copy()

# Examine distribution of each variable (including missings)
print(brfss_clean_df['addepev3'].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["firearm5"].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["loadulk2"].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["loadulk2"].value_counts(normalize = True, dropna = True))
print(brfss_clean_df['menthlth'].describe())
print(brfss_clean_df['sexvar'].value_counts(normalize = True, dropna = False))
print(brfss_clean_df['_age80'].describe())
print(brfss_clean_df["_racegr3"].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["primins1"].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["educa"].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["income3"].value_counts(normalize = True, dropna = False))
print(brfss_clean_df["children"].value_counts(normalize = True, dropna = False))

addepev3
0.0    0.790604
1.0    0.203426
NaN    0.005970
Name: proportion, dtype: float64
firearm5
NaN    0.827528
0.0    0.102755
1.0    0.069717
Name: proportion, dtype: float64
loadulk2
NaN    0.976821
1.0    0.012679
0.0    0.010500
Name: proportion, dtype: float64
loadulk2
1.0    0.546993
0.0    0.453007
Name: proportion, dtype: float64
count    425215.000000
mean          4.358383
std           8.320868
min           0.000000
25%           0.000000
50%           0.000000
75%           5.000000
max          30.000000
Name: menthlth, dtype: float64
sexvar
2    0.529723
1    0.470277
Name: proportion, dtype: float64
count    433323.000000
mean         55.375461
std          17.909598
min          18.000000
25%          41.000000
50%          58.000000
75%          71.000000
max          80.000000
Name: _age80, dtype: float64
_racegr3
1.0    0.722480
5.0    0.099893
2.0    0.075969
3.0    0.056207
4.0    0.023366
NaN    0.022085
Name: proportion, dtype: float64
primins1
1.0    0.4296

# ***Contingency Table (1) Prevalence of Depression among Gun Owners***

In [58]:
# Crosstab between depression and firearem ownership
depr_firearms_crosstab = pd.crosstab(brfss_clean_df['addepev3'], brfss_clean_df['firearm5'], margins = True)
print(depr_firearms_crosstab)

# Proportions
depr_firearms_prop = depr_firearms_crosstab = pd.crosstab(brfss_clean_df['addepev3'], brfss_clean_df['firearm5'], normalize = 'columns') * 100
print(depr_firearms_prop)

# P-value overall
chisq_pvalue = chi2_contingency(pd.crosstab(brfss_clean_df['addepev3'], brfss_clean_df['firearm5'])).pvalue
print(chisq_pvalue)

firearm5    0.0    1.0    All
addepev3                     
0.0       33848  23767  57615
1.0       10399   6296  16695
All       44247  30063  74310
firearm5        0.0        1.0
addepev3                      
0.0       76.497842  79.057313
1.0       23.502158  20.942687
2.487206276707508e-16


20.9% of gun owners have had depression compared with 23.5% of non-gun owners. The difference in depression between gun owners and non-owners is statistically significant at the 5% level (p < 0.001). 

# ***Logistic Regression (2a) Depression ~ Gun Ownership + covariates***

In [44]:
# Outcome: ADDEPEV3 (Depression)
# Main predictor: FIREARM5 (Gun ownership)
# Covariates: _AGE80, SEXVAR, _RACEGR3, EDUCA, INCOME3, PRIMINS1, 

# Logistic Regression
data_2a = brfss_clean_df[['addepev3', 'firearm5', '_age80', 'sexvar', '_racegr3', 'educa', 'income3', 'primins1']].dropna().copy() 
logreg_2a = 'addepev3 ~ C(firearm5) + _age80 + C(sexvar) + C(_racegr3) + C(educa) + C(income3) + C(primins1)'
model_2a = smf.logit(formula = logreg_2a, data = data_2a)
results_2a = model_2a.fit()

# Report AMEs
ame_2a = results_2a.get_margeff(at="overall", method="dydx", dummy="True")
print(ame_2a.summary())

Optimization terminated successfully.
         Current function value: 0.502693
         Iterations 6
        Logit Marginal Effects       
Dep. Variable:               addepev3
Method:                          dydx
At:                           overall
                        dy/dx    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
C(firearm5)[T.1.0]    -0.0076      0.003     -2.181      0.029      -0.014      -0.001
C(sexvar)[T.2]         0.1177      0.003     35.145      0.000       0.111       0.124
C(_racegr3)[T.2.0]    -0.1032      0.005    -21.631      0.000      -0.113      -0.094
C(_racegr3)[T.3.0]    -0.1039      0.006    -16.454      0.000      -0.116      -0.092
C(_racegr3)[T.4.0]     0.0296      0.012      2.506      0.012       0.006       0.053
C(_racegr3)[T.5.0]    -0.0935      0.005    -19.009      0.000      -0.103      -0.084
C(educa)[T.2.0]        0.0329      0.005      7.04

Owning a gun is associated with a 0.76 percentage point average decrease in the probability of having had depression, controlling for the other covariates. This difference is statistically significant at the 5% level (p = 0.029)

# ***OLS Regression (2b) Days of Poor Mental Health ~ Gun Ownership + covariates***

In [40]:
# Outcome: MENTLHLTH
# Main predictor: FIREARM5 (Gun ownership)
# Covariates: _AGE80, SEXVAR, _RACEGR3, EDUCA, INCOME3, PRIMINS1, 

# Logistic Regression
data_2b = brfss_clean_df[['menthlth', 'firearm5', '_age80', 'sexvar', '_racegr3', 'educa', 'income3', 'primins1']].dropna().copy() 
logreg_2b = 'menthlth ~ C(firearm5) + _age80 + C(sexvar) + C(_racegr3) + C(educa) + C(income3) + C(primins1)'
model_2b = smf.ols(formula = logreg_2b, data = data_2b)
results_2b = model_2b.fit()

print(results_2b.summary())

                            OLS Regression Results                            
Dep. Variable:               menthlth   R-squared:                       0.094
Model:                            OLS   Adj. R-squared:                  0.093
Method:                 Least Squares   F-statistic:                     381.6
Date:                Thu, 04 Dec 2025   Prob (F-statistic):               0.00
Time:                        17:56:05   Log-Likelihood:            -2.0747e+05
No. Observations:               59192   AIC:                         4.150e+05
Df Residuals:                   59175   BIC:                         4.151e+05
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             13.8588      0

Owning a gun is associated with an average decrease in poor mental health days of 0.22 days, controlling for the other covariates. This difference is statistically significant at the 5% level (p = 0.001). 

# ***Logistic Regression (3a) Unsafe Gun Storage Practices ~ Depression + covariates***

In [47]:
# Outcome: LOADULK2 (Unsafe storage)
# Main predictor: ADDEPEV3 (0/1 Depression)
# Covariates: _age80, sexvar, _racegr3, educa, income3, primins1, children

# keep only the respondents who have guns 
gun_df = brfss_clean_df[brfss_clean_df["firearm5"] == 1].copy()
print(gun_df["firearm5"].value_counts(dropna=False))

vars_3a = ["loadulk2", "addepev3", "_age80", "sexvar", "_racegr3", "educa", "income3", "primins1", "children"] #all vars needed for model 3a

m3a = gun_df[vars_3a].dropna().copy() # drop rows with any missings in these variables

#Logistic Regression
logreg_3a = "loadulk2 ~ addepev3 + _age80 + C(sexvar) + C(_racegr3) + C(educa) + C(income3) + C(primins1) + C(children)"

model_3a = smf.logit(formula=logreg_3a, data=m3a) #https://www.statsmodels.org/stable/generated/statsmodels.formula.api.logit.html
results_3a = model_3a.fit()

# Report AMEs
ame_3a = results_3a.get_margeff(at="overall", method="dydx", dummy="True")
print(ame_3a.summary())

firearm5
1.0    30210
Name: count, dtype: int64
Optimization terminated successfully.
         Current function value: 0.653516
         Iterations 5
        Logit Marginal Effects       
Dep. Variable:               loadulk2
Method:                          dydx
At:                           overall
                        dy/dx    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
C(sexvar)[T.2]        -0.1072      0.011     -9.493      0.000      -0.129      -0.085
C(_racegr3)[T.2.0]    -0.0505      0.020     -2.497      0.013      -0.090      -0.011
C(_racegr3)[T.3.0]    -0.0366      0.035     -1.055      0.292      -0.105       0.031
C(_racegr3)[T.4.0]    -0.0835      0.034     -2.443      0.015      -0.151      -0.017
C(_racegr3)[T.5.0]    -0.0909      0.025     -3.606      0.000      -0.140      -0.041
C(educa)[T.2.0]        0.0145      0.013      1.079      0.281      -0.012       0.041
C(

Among gun owners, being depressed is associated with a 2.1 percentage point average increase in the probability of unsafe gun storage practices, controlling for the other covariates. This difference is not statistically significant at the 5% level (p = 0.127)

# ***Logistic Regression (3b) Unsafe Gun Storage Practices ~ Days of Poor Mental Health + covariates***

In [48]:
#Outcome: loadulk2 (unsafe storage)
#main predictor: menthlth (continuous; # days of poor mental health)
#covariates: _age80, sexvar, _racegr3, educa, income3, primins1, children

# keep only the respondents who have guns 
gun_df = brfss_clean_df[brfss_clean_df["firearm5"] == 1].copy()
print(gun_df["firearm5"].value_counts(dropna=False))

vars_3b = ["loadulk2", "menthlth", "_age80", "sexvar", "_racegr3", "educa", "income3", "primins1", "children"] #all vars needed for model 3b

m3b = gun_df[vars_3b].dropna().copy() # drop rows with any missings in these variables

#Logistic Regression
logreg_3b = "loadulk2 ~ menthlth + _age80 + C(sexvar) + C(_racegr3) + C(educa) + C(income3) + C(primins1) + C(children)"

model_3b = smf.logit(formula=logreg_3b, data=m3b) #https://www.statsmodels.org/stable/generated/statsmodels.formula.api.logit.html
results_3b = model_3b.fit()

# Report AMEs
ame_3b = results_3b.get_margeff(at="overall", method="dydx", dummy="True")
print(ame_3b.summary())

firearm5
1.0    30210
Name: count, dtype: int64
Optimization terminated successfully.
         Current function value: 0.652627
         Iterations 5
        Logit Marginal Effects       
Dep. Variable:               loadulk2
Method:                          dydx
At:                           overall
                        dy/dx    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
C(sexvar)[T.2]        -0.1055      0.011     -9.420      0.000      -0.127      -0.084
C(_racegr3)[T.2.0]    -0.0540      0.020     -2.667      0.008      -0.094      -0.014
C(_racegr3)[T.3.0]    -0.0302      0.035     -0.863      0.388      -0.099       0.038
C(_racegr3)[T.4.0]    -0.0806      0.034     -2.349      0.019      -0.148      -0.013
C(_racegr3)[T.5.0]    -0.0913      0.025     -3.627      0.000      -0.141      -0.042
C(educa)[T.2.0]        0.0140      0.013      1.044      0.297      -0.012       0.040
C(

Among gun owners, a one-day increase in poor mental health days is associated with a 0.1 percentage point average increase in the probability of unsafe gun storage practices, controlling for the other covariates. This difference is not statistically significant at the 5% level (p = 0.132)

# ***Bar Charts of Average Predicted Probabilities and Average Number of Days by Gun Ownership, Depression, and Poor Mental Health Days (4a)***

# ***Map Unsafe Gun Storage Practices by State (4b)***

In [54]:
# keep only the respondents who have guns 
gun_df = brfss_clean_df[brfss_clean_df["firearm5"] == 1].copy()

vars_4b = ["loadulk2", "addepev3", "_age80", "sexvar", "_racegr3", "educa", "income3", "primins1", "children", "_state"] 

m4b = gun_df[vars_4b].dropna().copy() # drop rows with any missings in these variables

#Logistic Regression
logreg_4b = "loadulk2 ~ C(addepev3) + _age80 + C(sexvar) + C(_racegr3) + C(educa) + C(income3) + C(primins1) + C(children)"

model_4b = smf.logit(formula=logreg_4b, data=m4b) 
results_4b = model_4b.fit()

# Add predicted probabilities column
m4b['predicted_prob'] = results_4b.predict()
prob_dep_by_state = m4b.groupby('_state')['predicted_prob'].mean().reset_index()
print(prob_dep_by_state)

Optimization terminated successfully.
         Current function value: 0.653516
         Iterations 5
   _state  predicted_prob
0      AZ        0.551337
1      ID        0.537615
2      IN        0.530565
3      LA        0.524301
4      MI        0.540664
5      NJ        0.517584
6      NM        0.548577
7      NV        0.529030
8      OR        0.549205
9      SC        0.540967
10     VA        0.539215
11     VT        0.560987
12     WV        0.556851


In [56]:
# Sample data: State abbreviations and marginal effect
data = {
    'State': ['AZ', 'ID', 'IN', 'LA', 'MI', 'NJ', 'NM', 'NV', 'OR', 'SC', 'VA', 'VT', 'WV'],
    'Percent': [55.1, 53.8, 53.1, 52.4, 54.1, 51.8, 54.9, 52.9, 54.9, 54.1, 53.9, 56.1, 55.7]
}

df = pd.DataFrame(data)

# Create the choropleth map
fig = px.choropleth(df,
                    locations='State', # Column with state abbreviations
                    locationmode="USA-states", # Tells Plotly to use its built-in US state geometry
                    color='Percent', # Column with the metric to color by
                    scope="usa", # Limits the map to the USA
                    color_continuous_scale="RdBu_r", # Choose a color scale
                    color_continuous_midpoint=54.7,
                    title='Average Predicted Probability of Unsafe Gun Storage Among Gun Owners by State Relative to the Average of the Sample (54.7%)'
                   )
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show(renderer='browser')

There does not appear to be much geographic variation in the average predicted probability of unsafe gun storage across states. Among the 13 states, Vermont had the highest average predicted probability at 56.1% and New Jersey had the lowest at 51.8%. It is important to note that we are limited in our generalizability because we only have 13 states.  